# Present Value of Liabilities and Funding Ratio

This notebook discounts future cash obligations and measures asset coverage using a flat annual discount-rate assumption.

In [ ]:
import pandas as pd
import numpy as np
import risk_kit as rk

liabilities = pd.Series([1.0, 1.5, 2.0, 2.5], index=[3.0, 3.5, 4.0, 4.5], name='Liability')
liabilities.index.name = 'Years to payment'

if (liabilities < 0).any() or (liabilities.index <= 0).any():
    raise ValueError("This example expects positive future liabilities at positive payment times.")

## 1. Discount a future payment

A unit zero-coupon payment due at time $t$ is valued as $B(t)=(1+r)^{-t}$ under annual compounding and a flat annual rate.

In [ ]:
rate = 0.03
discount_factors = pd.Series(rk.discount(liabilities.index.to_numpy(), rate), index=liabilities.index, name='Discount factor')
pd.concat([liabilities, discount_factors, liabilities * discount_factors.rename('Discount factor')], axis=1)

## 2. Present value of a liability stream

The present value is the sum of each promised cash flow multiplied by its discount factor.

In [ ]:
nominal_total = liabilities.sum()
present_value = rk.pv(liabilities, rate)
print(f'Nominal liability total: {nominal_total:.3f}')
print(f'Present value:            {present_value:.3f}')
print(f'Discounting reduction:    {nominal_total-present_value:.3f}')

## 3. Funding ratio

The funding ratio is assets divided by the present value of liabilities. A ratio above 1 indicates assets exceed the modelled present value; below 1 indicates a shortfall.

In [ ]:
assets = 5.0
ratio = rk.funding_ratio(assets, liabilities, rate)
print(f'Funding ratio: {ratio:.2%}')

## 4. Sensitivity to the discount rate

For fixed positive future cash flows, a higher discount rate lowers present value and raises the measured funding ratio. This does not mean the economic risk of the liabilities has necessarily fallen.

In [ ]:
rates = np.linspace(0.0, 0.20, 101)  # Annual effective discount rates
assert assets > 0 and (liabilities > 0).all()
sensitivity = pd.DataFrame({'Discount rate': rates,
    'Liability present value': [rk.pv(liabilities, r) for r in rates]})
sensitivity['Funding ratio'] = assets / sensitivity['Liability present value']
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sensitivity['Discount rate'], sensitivity['Funding ratio'])
ax.set(title='Funding ratio sensitivity', xlabel='Annual discount rate', ylabel='Funding ratio')
ax.grid(alpha=.25)
plt.tight_layout()

## 5. Interactive rate and asset exploration (optional)

Interactive controls require ipywidgets; the static sensitivity analysis above is always available.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    print('Install ipywidgets to enable controls: pip install ipywidgets')
else:
    def show_funding_ratio(asset_value=5.0, annual_rate=0.03):
        pv_value = rk.pv(liabilities, annual_rate)
        print(f'Liability PV: {pv_value:.3f} | Funding ratio: {asset_value/pv_value:.2%}')
    display(widgets.interactive(show_funding_ratio,
        asset_value=widgets.FloatSlider(min=1, max=10, step=0.5, value=5),
        annual_rate=widgets.FloatSlider(min=0, max=0.20, step=0.005, value=0.03)))

## Assumptions and limitations

The flat curve and annual compounding convention are simplifications. Real liability valuation may require dated cash flows, term structures, inflation assumptions, credit spreads, optionality and matching asset cash flows. The funding ratio is only as reliable as those assumptions.


## Reproducibility and execution notes

- Run this notebook from the repository root after installing `requirements.txt`.
- The shared calculations are implemented in `risk_kit.py`; this notebook focuses on the research question and interpretation.
- Code cells are intentionally saved without stale execution outputs. Run **Kernel → Restart Kernel and Run All Cells** to reproduce the analysis from a clean state.
- Model parameters and historical sample choices are illustrative and should be changed only with the resulting assumptions documented.
